# Discrete and Batch Multi-Fidelity qMFKG Tutorial

This notebook demonstrates ordered numeric fidelity levels and qMFKG batches. The CSV log remains the source of truth, dry-run suggestions do not write, and append/observation transitions remain explicit.

In [ ]:
from pathlib import Path
import math
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib-cache'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge import CampaignSession

CONFIG_PATH = PROJECT_ROOT / 'configs' / '22_discrete_multi_fidelity_qmfkg.yaml'
SEED_LOG_PATH = PROJECT_ROOT / 'examples' / '22_discrete_multi_fidelity_qmfkg_campaign_log.csv'
WORKING_LOG_PATH = PROJECT_ROOT / 'examples' / '22_discrete_multi_fidelity_qmfkg_working_log.csv'
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / 'examples' / '22_discrete_multi_fidelity_qmfkg_latest_suggestions.csv'
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
TARGET_OBSERVED_ROWS = 15
shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)

## Validate and inspect fidelity coverage

The summary appends discrete-mode fields while preserving the continuous-fidelity field order.

In [ ]:
campaign.validate()
campaign.summary(), campaign.fidelity_summary(), campaign.next_action()

## Batch dry run and explicit campaign loop

qMFKG may select different configured fidelity levels in one batch. BoTorch constructs the discrete batch greedily with conditioning, then reports one joint post-selection acquisition value. Every row shares that value and the batch iteration, while predictions remain row-specific.

In [ ]:
def simulate_activity(row):
    loading = float(row['catalyst_loading'])
    temperature = float(row['reaction_temperature'])
    fidelity = float(row['fidelity'])
    target = 1.05 + 1.2 * loading - 2.1 * (loading - 0.34) ** 2
    temperature_term = -0.00012 * (temperature - 88.0) ** 2
    fidelity_bias = -0.7 * (1.0 - fidelity)
    variation = 0.03 * math.sin(7.0 * loading + 0.04 * temperature)
    return round(target + temperature_term + fidelity_bias + variation, 6)

while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    remaining = TARGET_OBSERVED_ROWS - len(campaign.observed_data())
    suggestions = campaign.suggest_next(batch_size=min(2, remaining))
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions['row_id']:
        row = campaign.df.loc[campaign.df['row_id'] == row_id].iloc[0]
        campaign.mark_observed(row_id=row_id, objective_value=simulate_activity(row))

campaign.fidelity_summary()

## Export read-only diagnostics

Discrete fidelity diagnostics show exact configured ticks and include levels with zero observations.

In [ ]:
campaign.export_report(REPORTS_DIR / '22_discrete_multi_fidelity_qmfkg_report.txt')
campaign.plot_progress(save_path=REPORTS_DIR / '22_discrete_multi_fidelity_qmfkg_progress.png')
campaign.plot_fidelity_diagnostics(save_path=REPORTS_DIR / '22_discrete_multi_fidelity_qmfkg_fidelity_diagnostics.png')

## Limitations

Fidelity levels are ordered numeric values on one continuous variable. Named sources, per-level cost tables, multi-objective fidelity, contextual fidelity, structured fidelity, replicate-aware fidelity, and noisy multi-fidelity acquisitions remain deferred. Batch size is limited to four.